# Phase A: Wrist-normalized landmarks and SVM

Extract MediaPipe Tasks hand landmarks from `data/personal/<LETTER>/*.jpg`, evaluate an RBF SVM with a stratified 20% holdout, then fit and save the final model. Run from the project root.

In [ ]:
from pathlib import Path
import pandas as pd
from src.landmark_svm import (
    DEFAULT_DATA_DIR, DEFAULT_MODEL_PATH, create_hand_landmarker,
    evaluate_svm, fit_svm, load_landmark_dataset, save_svm_model,
)

The collector downloads the hand-landmarker model to the user cache. If it is missing, run `python collect_data.py` once with an internet connection.

In [ ]:
with create_hand_landmarker() as hand_landmarker:
    dataset = load_landmark_dataset(DEFAULT_DATA_DIR, hand_landmarker)

features, labels = dataset.features, dataset.labels
print(f'{len(labels)} images; {features.shape[1]} features per image')
print(f'{len(dataset.rejected)} images rejected')
display(pd.DataFrame(dataset.rejected, columns=['image', 'reason']).head(20))
display(pd.Series(labels).value_counts().sort_index().to_frame('samples'))

## Stratified holdout evaluation

Evaluation trains on 80% of each class and reports held-out accuracy, per-class metrics, and the confusion matrix.

In [ ]:
evaluation = evaluate_svm(features, labels, test_size=0.2, random_state=42)
print(f'Holdout accuracy: {evaluation.accuracy:.3f}')
display(pd.DataFrame(evaluation.classification_report).T)
display(pd.DataFrame(
    evaluation.confusion_matrix,
    index=evaluation.classes,
    columns=evaluation.classes,
))

## Fit and save final model

After reviewing the holdout metrics, fit on all collected samples. The output is ignored by git under `models/`.

In [ ]:
classes = sorted(set(labels.tolist()))
final_model = fit_svm(features, labels)
model_path = save_svm_model(final_model, classes, DEFAULT_MODEL_PATH)
print(f'Saved model: {model_path}')